In [6]:
# import library
import pandas as pd

In [7]:
# import dataset
file_path = "Group 8- original dataset-Electronic_sales_Sep2023-Sep2024.csv"
sale_data = pd.read_csv(file_path)

In [8]:
# data preprocessing & cleaning
# transfer date form
sale_data["Purchase Date"] = pd.to_datetime(sale_data["Purchase Date"], errors="coerce")

# delete "cancelled" orders
completed_rows = [] #create a new row to hold all completed oreders

for i in range(len(sale_data)):
    row = sale_data.iloc[i] #get current row
    
    # identify whether current row is completed
    if row["Order Status"] == "Completed":
        completed_rows.append(row)

# complete the selection
sales_cleaned = pd.DataFrame(completed_rows)

In [11]:
# create a new set to caculate
rfm_data = []

# get the unique customer id
customers = sales_cleaned["Customer ID"].unique()

# calculate each customer's R, F, M; identify the calculation formulas as below:
## R = last purchase date - first purchase date;
## F = purchase times (count customer ID);
## M = total purchase cost (sum of total price of each customer).

for cid in customers:
    customer_data = sales_cleaned[sales_cleaned["Customer ID"] == cid]
    
    # calculate R
    first_date = customer_data["Purchase Date"].min()
    last_date = customer_data["Purchase Date"].max()
    R = (last_date - first_date).days
    
    # calculate F
    F = len(customer_data)
    
    # calculate M
    M = customer_data["Total Price"].sum()
    
    # put the results into rfm_data
    rfm_data.append([cid, R, F, M])

In [15]:
# transfer to pandas DataFrame
customer_analysis = pd.DataFrame(rfm_data, columns=["Customer ID", "R", "F", "M"])

# set score methods for R, F, M, the rules are identified as below (compare each single customer with population):
## if R_single < average R_population, assign it with 1, otherwise assign with 0;
## if F_single > average F_population, assign it with 1, otherwise assign with 0;
## if M_single > median M_population, assign it with 1, otherwise assign with 0:
### the reason why we choose median_M instead of average_M is for better and more significant categorizing

# calculate the average/median threshold
R_mean = customer_analysis["R"].mean()
F_mean = customer_analysis["F"].mean()
M_median = customer_analysis["M"].median()

# score for each customer
R_score_list = []
F_score_list = []
M_score_list = []
RFM_value_list = []

for i in range(len(customer_analysis)):
    r = customer_analysis.loc[i, "R"]
    f = customer_analysis.loc[i, "F"]
    m = customer_analysis.loc[i, "M"]
    
    # score R
    if r < R_mean:
        R_score = 1
    else:
        R_score = 0
    
    # score F
    if f > F_mean:
        F_score = 1
    else:
        F_score = 0
    
    # socre M
    if m > M_median:
        M_score = 1
    else:
        M_score = 0
    
    # joint R, F, M
    RFM_value = str(R_score) + str(F_score) + str(M_score)
    
    # store the results
    R_score_list.append(R_score)
    F_score_list.append(F_score)
    M_score_list.append(M_score)
    RFM_value_list.append(RFM_value)

# add results to final dataset
customer_analysis["R_score"] = R_score_list
customer_analysis["F_score"] = F_score_list
customer_analysis["M_score"] = M_score_list
customer_analysis["RFM_value"] = RFM_value_list

print(customer_analysis.head())

   Customer ID    R  F         M  R_score  F_score  M_score RFM_value
0         1000    0  1    741.09        1        0        0       100
1         1002  297  2   5020.60        0        1        1       011
2         1003    0  1     41.50        1        0        0       100
3         1004    0  1     83.00        1        0        0       100
4         1005  146  2  11779.11        0        1        1       011
